# Student Concentration HAR — Google Colab

1. **Runtime → Change runtime type → GPU**
2. Sửa **toàn bộ biến** trong cell **CẤU HÌNH HARDCODE** (Kaggle token, slug, đường dẫn…)
3. **Runtime → Run all**

Không dùng Colab Secrets / file `.env` — mọi thứ nằm trong notebook.

In [ ]:
# ============================================================
# CẤU HÌNH HARDCODE — sửa các giá trị bên dưới rồi Run all
# ============================================================

# --- Nguồn code: "git" hoặc "drive" ---
SOURCE = "git"
REPO_URL = "https://github.com/YOUR_USER/student-concentration-har.git"
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/student-concentration-har"

# --- Kaggle (bắt buộc nếu chưa có dataset/ trên máy Colab) ---
KAGGLE_API_TOKEN = "PASTE_KAGGLE_TOKEN_HERE"
KAGGLE_DATASET_SLUG = "owner/ten-bo-du-lieu"

# --- Train / evaluate ---
MODELS_TO_TRAIN = ["lrcn"]  # "convlstm", "movinet"
RUN_EVALUATE = True

# --- Lưu kết quả lên Drive ---
SAVE_TO_DRIVE = True
DRIVE_RESULTS_PATH = "/content/drive/MyDrive/student-concentration-har-results"

# ============================================================
import os

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG

print("SOURCE:", SOURCE)
print("KAGGLE_DATASET_SLUG:", KAGGLE_DATASET_SLUG)
print("MODELS:", MODELS_TO_TRAIN)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")


def find_project_on_drive(start: Path) -> Path | None:
    """Tìm thư mục project (có src/training/train_lrcn.py)."""
    if not start.is_dir():
        return None
    direct = start / "student-concentration-har"
    if (direct / "src/training/train_lrcn.py").is_file():
        return direct
    if (start / "src/training/train_lrcn.py").is_file():
        return start
    for train_script in start.rglob("train_lrcn.py"):
        root = train_script.parent.parent.parent
        if (root / "requirements.txt").is_file() and (root / "src/utils/dataset.py").is_file():
            return root
    return None


def resolve_drive_project(path_str: str) -> Path:
    from google.colab import drive

    if not Path("/content/drive").is_dir():
        drive.mount("/content/drive")

    if path_str.strip():
        project = Path(path_str)
    else:
        project = find_project_on_drive(MYDRIVE)

    if project is None or not project.is_dir():
        print("Không tìm thấy project. Nội dung MyDrive (cấp 1):")
        if MYDRIVE.is_dir():
            for item in sorted(MYDRIVE.iterdir())[:30]:
                print(" ", item.name + ("/" if item.is_dir() else ""))
        raise FileNotFoundError(
            "Không thấy project trên Drive.\n"
            "Cách 1: Upload folder 'student-concentration-har' vào MyDrive (gốc hoặc subfolder)\n"
            "Cách 2: Điền đúng DRIVE_PROJECT_PATH, ví dụ:\n"
            "  /content/drive/MyDrive/Colab Notebooks/student-concentration-har\n"
            "Cách 3: Đổi SOURCE='git' + REPO_URL (không cần Drive cho code)"
        )
    return project


SOURCE = SOURCE.lower().strip()
if SOURCE == "drive":
    PROJECT_ROOT = resolve_drive_project(DRIVE_PROJECT_PATH)
elif SOURCE == "git":
    if not REPO_URL.strip():
        raise ValueError("SOURCE='git' cần REPO_URL. Hoặc dùng SOURCE='drive'.")
    name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    dest = Path("/content") / name
    if dest.is_dir():
        shutil.rmtree(dest)
    subprocess.run(["git", "clone", REPO_URL, str(dest)], check=True)
    PROJECT_ROOT = dest
else:
    raise ValueError("SOURCE phải là 'git' hoặc 'drive'")

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
!ls -la

In [ ]:
# Bootstrap: secrets Kaggle + pip + dataset + smoke test
!python scripts/colab_setup.py

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src" / "utils"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "models"))

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# === TRAIN ===
TRAIN_SCRIPTS = {
    "lrcn": "src/training/train_lrcn.py",
    "convlstm": "src/training/train_convlstm.py",
    "movinet": "src/training/train_movinet.py",
}

for name in MODELS_TO_TRAIN:
    script = TRAIN_SCRIPTS.get(name)
    if not script:
        raise ValueError(f"Model không hỗ trợ: {name}")
    print(f"\n{'='*50}\nTrain {name.upper()}\n{'='*50}")
    !python {script}

In [ ]:
# === EVALUATE (test set) ===
if RUN_EVALUATE:
    for name in MODELS_TO_TRAIN:
        print(f"\n--- Evaluate {name} ---")
        !python src/utils/evaluate.py --model {name}
else:
    print("Bỏ qua evaluate (RUN_EVALUATE=False)")

In [ ]:
# === Lưu kết quả sang Google Drive ===
from pathlib import Path
import shutil

results = Path("results")
if SAVE_TO_DRIVE and results.is_dir():
    from google.colab import drive
    if not Path("/content/drive").is_dir():
        drive.mount("/content/drive")
    dest = Path(DRIVE_RESULTS_PATH)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(results, dest)
    print("Đã copy results ->", dest)
else:
    print("Kết quả trong:", results.resolve())
    !find results -type f 2>/dev/null | head -20